# Player Analysis - Batch Processing - Gold Layer

At the Gold layer, data is primed and refined to answer the domain questions:
- Who are the players that scored the maximum goals?
- Who are the players with the least number of bookings?
- Who has the maximum average goal scoring throughout the tournament?
- Who are the players that have scored the maximum number of goals in penalty shootouts?
- Who are the players that got substituted the most?
- Who are the players that were substituted the most in a tournament?

In [0]:
%pip uninstall -y databricks_helpers
%pip install git+https://github.com/data-derp/databricks_helpers#egg=databricks_helpers

In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]
username = current_user.split("@")[0]  # e.g. mythra.varun
working_directory = f"/FileStore/{username}/player_analysis_batch_processing/gold"
silver_working_directory = f"/FileStore/{username}/player_analysis_batch_processing/silver"
bronze_working_directory = f"/FileStore/{username}/player_analysis_batch_processing/bronze"
silver_output = f"{silver_working_directory}/output"
bronze_output = f"{bronze_working_directory}/output"

print(f"current_user       : {current_user}")
print(f"silver_output      : {silver_output}")
print(f"bronze_output      : {bronze_output}")

In [0]:
import pyspark.sql.functions as F

goals_per_player_per_tournament_df         = spark.read.parquet(f"{silver_output}/goals_per_player_per_tournament")
bookings_per_player_per_tournament_df      = spark.read.parquet(f"{silver_output}/bookings_per_player_per_tournament")
substitutions_per_player_per_tournament_df = spark.read.parquet(f"{silver_output}/substitutions_per_player_per_tournament")
penalty_kicks_per_player_per_tournament_df = spark.read.parquet(f"{silver_output}/penalty_kicks")
matches_df                                 = spark.read.parquet(f"{silver_output}/matches")
total_goals_per_player_df                  = spark.read.parquet(f"{silver_output}/goals")
total_bookings_per_player_df               = spark.read.parquet(f"{silver_output}/bookings")
total_substitutions_per_player_df          = spark.read.parquet(f"{silver_output}/total_substitutions")
total_penalty_kicks_per_player_df          = spark.read.parquet(f"{silver_output}/total_penalty_kicks")

squads_bronze_df = spark.read.parquet(f"{bronze_output}/squads").select("player_id", "tournament_id", "position_name", "position_code")

print("All Silver DFs loaded.")

The data below is primed to answer the question:
## Who are the top scorers in each FIFA tournament from 1930 to 2022?
- We do this by ranking the top 5 players who scored the most goals.
- We also look at the trend of this ranking over the last 94 years.
- Due to World War II, FIFA canceled the tournaments originally planned for 1942 and 1946 because of the global conflict and its widespread devastation.
- Following the end of the war in 1945, the international community required several years to rebuild its infrastructure and economies, allowing the tournament to finally resume in 1950 in Brazil.

## Gold Layer — Derived Insights from Silver
Enriched, ranked, and aggregated views ready for BI and reporting:
- **Top scorers per tournament**: ranked top 10 goal scorers per tournament
- **Tournament summary**: cross-domain aggregate stats (goals, bookings, substitutions, matches) per tournament
- **Player career summary**: combined all-time stats per player across goals, bookings, substitutions and penalties

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

def top_scorers_per_tournament(source: DataFrame, n: int = 10) -> DataFrame:
    window = Window.partitionBy("tournament_id").orderBy(F.desc("num_goals"))
    return (
        source
        .withColumn("rank", F.rank().over(window))
        .filter(F.col("rank") <= n)
        .withColumn("player_name",
            F.when(F.col("given_name") == "not applicable", F.col("family_name"))
             .otherwise(F.concat(F.col("given_name"), F.lit(" "), F.col("family_name"))))
        .select("tournament_id", "rank", "player_name", "team_name", "num_goals", "position_code")
        .orderBy(F.desc("tournament_id"), F.asc("rank"))
    )

gold_top_scorers_df = goals_per_player_per_tournament_df.transform(top_scorers_per_tournament, 5)
display(gold_top_scorers_df.limit(30))
display(gold_top_scorers_df.count())

In [0]:
# Latest position per player — from their most recent tournament
player_latest_position_df = (
    squads_bronze_df
    .withColumn("rn", F.row_number().over(Window.partitionBy("player_id").orderBy(F.desc("tournament_id"))))
    .filter(F.col("rn") == 1)
    .select("player_id", "position_name", "position_code")
)

gold_player_career_summary_df = (
    total_goals_per_player_df
    .withColumnRenamed("num_goals", "career_goals")
    .join(
        total_bookings_per_player_df
        .select("player_id", "team_name", "num_bookings", "yellow_cards", "red_cards")
        .withColumnRenamed("num_bookings", "career_bookings"),
        ["player_id", "team_name"], "left")
    .join(
        total_substitutions_per_player_df
        .select("player_id", "team_name", "times_subbed_on", "times_subbed_off")
        .withColumnRenamed("times_subbed_on", "career_subs_on")
        .withColumnRenamed("times_subbed_off", "career_subs_off"),
        ["player_id", "team_name"], "left")
    .join(
        total_penalty_kicks_per_player_df
        .select("player_id", "team_name", "penalty_shootout_goals", "num_penalty_kicks"),
        ["player_id", "team_name"], "left")
    .join(player_latest_position_df, "player_id", "left")
    .fillna(0, subset=["career_goals", "career_bookings", "yellow_cards", "red_cards",
                       "career_subs_on", "career_subs_off", "penalty_shootout_goals", "num_penalty_kicks"])
    .withColumn("player_name",
        F.when(F.col("given_name") == "not applicable", F.col("family_name"))
         .otherwise(F.concat(F.col("given_name"), F.lit(" "), F.col("family_name"))))
    .orderBy(F.desc("career_goals"))
)
display(gold_player_career_summary_df.limit(20))
display(gold_player_career_summary_df.count())

In [0]:
CATALOG = dbutils.widgets.get("catalog")
if CATALOG != "hive_metastore":
    spark.sql(f"USE CATALOG {CATALOG}")
SCHEMA_NAME = "player_analysis"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}")

gold_top_scorers_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_NAME}.gold_top_scorers_per_tournament")

gold_player_career_summary_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_NAME}.gold_player_career_summary")

print(f"Gold tables written to UC:")
print(f"  {CATALOG}.{SCHEMA_NAME}.gold_top_scorers_per_tournament")
print(f"  {CATALOG}.{SCHEMA_NAME}.gold_player_career_summary")